# Certificate errors on a corporate laptop

Community contribution by Rutger Rienks. This documents an optional workaround that worked on my macOS work laptop when connecting to OpenRouter from the [setup notebook](../../setup/setup.ipynb). It is not a general recommendation for corporate machines.

An `APIConnectionError` alone is not enough to identify a certificate problem. Check the underlying error for `CERTIFICATE_VERIFY_FAILED` or `self-signed certificate in certificate chain`. A related `uv` download error may say `UnknownIssuer`.

Some corporate proxies inspect HTTPS traffic using a company certificate. The operating system may trust it while Python's separate certificate bundle does not. On my machine, `SSL_CERT_FILE` also pointed to a separate `certifi` bundle. The default Python connection failed, while a connection using macOS certificate verification succeeded.

## Check with IT first

Corporate setups differ. Follow your IT team's approved certificate configuration and the course's troubleshooting guidance first. Do not use this workaround to bypass company policy or network restrictions.

- `truststore.inject_into_ssl()` changes SSL-context creation process-wide, not just for the OpenRouter client. It can affect other libraries in the same notebook kernel.
- The cell removes `SSL_CERT_FILE` and `SSL_CERT_DIR` overrides from the current process. Those may be intentional company settings; do not remove them without approval.
- Certificate verification stays enabled, using the operating system's trust decisions. No certificates are deleted, and terminal or system settings are not changed.
- This was tested on macOS with Python 3.12. It is not verified for every operating system, library or corporate environment. If it fails, ask IT rather than disabling certificate verification.

## Optional workaround for a Python notebook

1. Select the project's `.venv` kernel. If `truststore` is missing, run `uv pip install truststore` in a terminal at the repository root, using your approved certificate settings. This installs it locally without adding a project dependency.
2. After confirming this approach is appropriate with IT, copy the code cell below into the **affected notebook**, before its imports. Running it in this separate notebook does not fix another notebook's kernel.
3. Set `USE_SYSTEM_CERTIFICATES = True` in that copied cell, restart the affected notebook's kernel, then run the cell before importing OpenAI or creating any HTTP clients. Continue with the normal notebook cells.

The cell is disabled by default so **Run All** does not activate it accidentally. To undo activation, set the option back to `False` and restart the kernel. Running the disabled cell alone does not restore the previous SSL configuration.

In [ ]:
USE_SYSTEM_CERTIFICATES = False

if USE_SYSTEM_CERTIFICATES:
    import os
    import truststore

    os.environ.pop("SSL_CERT_FILE", None)
    os.environ.pop("SSL_CERT_DIR", None)
    truststore.inject_into_ssl()
    print("System certificate verification enabled for this notebook process.")

## Separate issue: certificate errors during `uv sync`

The Python cell above does not change `uv`: it is a separate program. With a recent version of `uv`, `uv sync --system-certs` uses the operating system's certificate verification. A configured `SSL_CERT_FILE` or `SSL_CERT_DIR` can override that certificate source, so ask IT to check those settings if the error remains.

References: [uv TLS certificates](https://docs.astral.sh/uv/concepts/authentication/certificates/) and [truststore documentation](https://truststore.readthedocs.io/en/latest/).